In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor
import os
import base64
import json
import re
#from huggingface_hub import login
#from transformers import AutoTokenizer, AutoModel
from langchain_google_genai import ChatGoogleGenerativeAI
#from huggingface_hub import InferenceClient
#from diffusers import StableDiffusionPipeline
load_dotenv()


True

In [2]:
with open("Aladdin_Adventures.png", "rb") as f:
    image_b64= base64.b64encode(f.read()).decode()

#image_b64[:200]

@tool
def storyImagePrompt(prompt: str):
    """extract only the image generation prompt and save it to a text file."""
    with open("story_image.txt", "w", encoding="utf-8") as f:
        f.write(prompt)
    return prompt

print(storyImagePrompt.name)



storyImagePrompt


In [3]:
#llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")
llm = ChatGroq(model="qwen/qwen3.8-27b")

messages= HumanMessage(content=[{"type":"image_url", "image_url":{"url": f"data:image/png;base64,{image_b64}"}},
                                {"type":"text", "text": """Analyse this story image. 
                                Create one 'colorful image generation prompt' which shows main characters and justifies moral of the story".
                                Return only valid JSON"""
                                }
                                ])

#print(messages.content)

In [4]:
agent= create_agent(
    llm,
    tools=[storyImagePrompt],
    system_prompt= """when a user ask for story image prompt, use storyImagePrompt. The generated output must not contain any words that cause third party copyright conflicts.
                    You have to strictly fetch all info from the user provided image only. Do not provide response if storyImagePrompt is not called.
                    """
)

def run(message: list):
    #config= {"configurable":{"thread_id":"user_aashu_session_id"}}
    response= agent.invoke({"messages": [{"role":"user", "content": message}]}) 
    print(response["messages"][-1])

run(messages.content)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3.8-27b` in organization `org_01m3tttshnej0a4z3k84ppj9cm` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 200000, Requested 3162. Please try again in 22m45.984s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [92]:
#result= response['messages']
#print(response['messages'][-1].response_metadata)
print(re.sub(r'\s+', ' ', messages.content[-1]["text"]).strip())


{'token_usage': {'completion_tokens': 305, 'prompt_tokens': 2339, 'total_tokens': 2644, 'completion_time': 0.674255748, 'completion_tokens_details': None, 'prompt_time': 0.183757941, 'prompt_tokens_details': None, 'queue_time': 0.497008813, 'total_time': 0.858013689}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}
Analyse this story image. You have to prepare a list of 2 questions and their answers that can be created from within this story. Create one 'colorful image generation prompt' which shows main characters and moral of the story". Return only valid JSON
